In [1]:
from google.colab import files

# This will prompt you to select a file from your computer
uploaded = files.upload()

Saving placementt.csv to placementt.csv


In [3]:
# ==============================================================================
# BuildX Analytics Project: Part B1 - Data Cleaning & Preparation
# Dataset: Factors Affecting Campus Placement
# Expected Output File: cleaned_dataset.csv
# ==============================================================================

# ------------------------------------------------------------------------------
# STEP 1: Loading & Exploring the Dataset (1 Mark)
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np

# Load the raw dataset using the pandas read_csv function
# Replace 'Placement_Data.csv' with your local path or Kaggle file name
df = pd.read_csv('placementt.csv')

# Display the first 5 records to verify structure, headers, and values
print("--- INSIGHT: FIRST 5 ROWS OF THE DATASET ---")
print(df.head(5))

# Display structural summary of columns, data shapes, and default system types
print("\n--- DATASET STRUCTURAL SUMMARY ---")
df.info()

# ------------------------------------------------------------------------------
# STEP 2: Checking & Handling Missing Values (2 Marks)
# ------------------------------------------------------------------------------
# Evaluate the exact count of missing (Null/NaN) values across every database column
print("\n--- INITIAL MISSING VALUE COUNTS ---")
print(df.isnull().sum())

# CRITICAL ANALYTICAL CHOICE EXPLANATION:
# The 'salary' column contains several missing values. Looking at the dataset,
# these null entries map exactly to students whose placement_status is 'Not Placed'.
# Dropping these rows would destroy valuable data about unplaced candidates.
# Instead, we fill these null values with 0.0 to reflect their actual placement income.
if 'salary' in df.columns:
    df['salary'] = df['salary'].fillna(0.0)

# Check to ensure all structural missing values have been successfully handled
print("\n--- VERIFYING MISSING VALUES AFTER TREATMENT ---")
print(df.isnull().sum())

# ------------------------------------------------------------------------------
# STEP 3: Removing Duplicate Records (1 Mark)
# ------------------------------------------------------------------------------
# Calculate the total number of exact duplicate rows present in the frame
duplicate_count = df.duplicated().sum()
print(f"\nTotal duplicate rows detected: {duplicate_count}")

# Remove duplicates permanently while preserving the original index structure
if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicate rows successfully expunged from the active dataset.")

# ------------------------------------------------------------------------------
# STEP 4: Cleaning Text Columns & Fixing Incorrect Data Types (2 Marks)
# ------------------------------------------------------------------------------
# 1. Standardize all column names to clean, lowercase snake_case to prevent syntax mismatches
df.columns = df.columns.str.strip().str.lower()
print("\nStandardized Column Headers:\n", df.columns.tolist())

# 2. Loop through all text-based categorical columns to strip trailing white spaces
# and convert strings into standard 'Title Case' format for dashboard uniformity
categorical_text_columns = ['gender', 'ssc_b', 'hsc_b', 'hsc_s', 'degree_t', 'workex', 'specialisation', 'status']

for column in categorical_text_columns:
    if column in df.columns:
        df[column] = df[column].astype(str).str.strip().str.title()

# 3. Explicitly fix data types. Ensure sl_no is integer, and currency fields are floating decimals
if 'sl_no' in df.columns:
    df['sl_no'] = df['sl_no'].astype(int)
if 'salary' in df.columns:
    df['salary'] = df['salary'].astype(float)

print("\nData validation and type casting complete.")

# ------------------------------------------------------------------------------
# STEP 5: Creating at Least One New Useful Column (2 Marks)
# ------------------------------------------------------------------------------
# Business Rule Logic: We need to categorize students into distinct 'Performance Brackets'
# based on their undergraduate degree scores (degree_p). This feature allows our
# SQL scripts and Power BI charts to group students into actionable academic tiers.

def calculate_academic_bracket(percentage):
    if percentage >= 75.0:
        return 'High Distinction'
    elif percentage >= 60.0:
        return 'First Class'
    else:
        return 'Pass Grade'

# Apply the custom logic row-by-row to populate the new feature column
df['performance_bracket'] = df['degree_p'].apply(calculate_academic_bracket)

print("\n--- NEW COLUMN SNEAK PEEK (First 5 Rows) ---")
print(df[['degree_p', 'performance_bracket']].head())

# ------------------------------------------------------------------------------
# STEP 6: Exporting the Cleaned Dataset (1 Mark)
# ------------------------------------------------------------------------------
# Save the final transformed dataframe into a clean CSV file without the index column
df.to_csv('cleaned_dataset.csv', index=False)
print("\nSUCCESS: 'cleaned_dataset.csv' exported and ready for SQL and Power BI import!")

--- INSIGHT: FIRST 5 ROWS OF THE DATASET ---
   sl_no gender  ssc_p    ssc_b  hsc_p    hsc_b     hsc_s  degree_p  \
0      1      M  67.00   Others  91.00   Others  Commerce     58.00   
1      2      M  79.33  Central  78.33   Others   Science     77.48   
2      3      M  65.00  Central  68.00  Central      Arts     64.00   
3      4      M  56.00  Central  52.00  Central   Science     52.00   
4      5      M  85.80  Central  73.60  Central  Commerce     73.30   

    degree_t workex  etest_p specialisation  mba_p      status    salary  
0   Sci&Tech     No     55.0         Mkt&HR  58.80      Placed  270000.0  
1   Sci&Tech    Yes     86.5        Mkt&Fin  66.28      Placed  200000.0  
2  Comm&Mgmt     No     75.0        Mkt&Fin  57.80      Placed  250000.0  
3   Sci&Tech     No     66.0         Mkt&HR  59.43  Not Placed       NaN  
4  Comm&Mgmt     No     96.8        Mkt&Fin  55.50      Placed  425000.0  

--- DATASET STRUCTURAL SUMMARY ---
<class 'pandas.core.frame.DataFrame'>
Rang